In [5]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

In [30]:
def get_ingredients(url):
    response = requests.get(url)
    # check if the URL has been redirected to determine end of letter
    if response.url != url:
        # empty list and a flag indicating redirection
        return [], True  

    soup = BeautifulSoup(response.content, 'html.parser')
    ingredients = []

    # scrape ingredients based on pattern
    for h3 in soup.find_all('h3', class_=lambda x: x and 'promo__title' in x):
        ingredient_name = h3.get_text(strip=True)
        ingredients.append(ingredient_name)
    
    return ingredients, False

def scrape_ingredients():
    base_url = "https://www.bbc.co.uk/food/ingredients/a-z/"
    ingredients = []

    # loop through letters a to z
    for letter in range(ord('a'), ord('z') + 1):
        page = 1
        while True:
            url = f"{base_url}{chr(letter)}/{page}"
            print(f"Scraping URL: {url}")
            current_ingredients, redirected = get_ingredients(url)

            if redirected or not current_ingredients:
                # if redirected or no ingredients are found, break to the next letter
                break
            
            ingredients.extend(current_ingredients)
            page += 1
            time.sleep(0.5)  

    return ingredients

if __name__ == "__main__":
    all_ingredients = scrape_ingredients()
    print(f"Total ingredients scraped: {len(all_ingredients)}")
    for ingredient in all_ingredients:
        print(ingredient)

Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/a/1
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/a/2
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/a/3
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/b/1
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/b/2
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/b/3
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/b/4
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/b/5
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/b/6
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/c/1
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/c/2
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/c/3
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/c/4
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/c/5
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/c/6
Scraping URL: https://www.bbc.co.uk/food/ingredients/a-z/c/7
Scraping URL: https://ww

In [48]:
import unicodedata
import inflect
p = inflect.engine()
def singularize(word):
    return p.singular_noun(word) or word 

cleaned_all_ingredients = []

for i in all_ingredients:
    word = unicodedata.normalize('NFKD', i).encode('ascii','ignore').lower().decode('ascii')
    cleaned_all_ingredients.append(singularize(word))
    
cleaned_all_ingredients = [s.replace('chilly', 'chili') for s in cleaned_all_ingredients]

In [52]:
pd.DataFrame(cleaned_all_ingredients,columns=['ingr_name']).to_csv('ingredients.csv')